# 🏗️ Lakehouse & Historic Analysis — Complete Reference (`ws_mch_iot`)

**Machinery IoT — cold batch medallion (Flow B).** The eventhouse owns the *hot* path; the lakehouse owns
the *historic* path: a scheduled, incremental, idempotent **medallion** (bronze → silver → gold) that turns
the cleansed live telemetry into a **Direct Lake** star schema for Power BI history/trend analysis. Everything
in this folder, end to end, nothing left out.

```
┌── EVENTHOUSE (Flow A, hot) ──────────────┐
│  kdb_mch.tbl_mch_clean                    │  classified telemetry (NORMAL/ALERT/ANOMALY/BAD/NULL,
│   • good rows  ───────────────┐           │  is_late, lag_sec, ingest_ts = ingestion_time())
│   • BAD / NULL rows  ─► stay here          │  rejects remain queryable via fn_mch_errors()
└────────────────────────────────┼──────────┘
                                 ▼  SCHEDULED incremental pull (ingest_ts watermark) + BAD/NULL filter
                          brz.evt  (lean bronze — append-only, audit col bronze_load_ts)
                                 ▼  incremental HWM on event_ts (24h re-scan grace) + idempotent MERGE on event_id
                          slv.evt  ──(rejects)──► slv.dlq   (NULL/BAD/UNKNOWN_REF/SCHEMA/STALE)
                                 ▼  cross-store join to wh_mch.dim.* CURRENT rows (scd_is_current=1)
                          gld.ft_telemetry  +  gld.agg_machine_daily
                                 ▼  Direct Lake (DirectLakeOnly)
                          sm_mch_historic (semantic model) ──► rpt_mch_historic (Power BI report)

ctl.wm (watermarks) · ctl.freshness (SLA log)   |   3 pipelines (master→ingest→medallion) + Daily 00:00 UTC schedule
```

| Item | Type | Role |
|---|---|---|
| `lh_mch` | Lakehouse (schema-enabled) | medallion store + notebook OneLake I/O. Tables/`<schema>`/`<table>` |
| `nb_mch_lh_ingest` | Notebook | bronze ingest (incremental, scheduled, BAD/NULL filter) |
| `nb_mch_lh_bronze_silver` | Notebook | bronze→silver incremental MERGE + DLQ |
| `nb_mch_lh_gold` | Notebook | silver→gold cross-store dim join |
| `nb_mch_lh_maint` | Notebook | OPTIMIZE (compact) + VACUUM |
| `nb_mch_lh_freshness` | Notebook | gold freshness SLA → ctl.freshness + delay alert |
| `nb_mch_lh_orchestrator` | Notebook | Flow-B chain driver + backfill_hours param |
| `nb_mch_hist_model` | Notebook | builds Direct Lake `sm_mch_historic` + measures |
| `nb_mch_hist_model + report builder` | Notebook | builds/documents `rpt_mch_historic` |
| `nb_mch_lh_*_spark` | Notebooks (4) | PySpark / Spark-SQL **presentation** variants (gated) |
| `sm_mch_historic` | Semantic model | Direct Lake star over gold |
| `rpt_mch_historic` | Power BI report | historic / trend analysis |
| `pl_mch_lh_master` / `_ingest` / `_medallion` | Data Pipelines (3) | orchestration + Daily 00:00 UTC schedule |
| `df_mch_dimrefresh` | Dataflow Gen2 | low-code dim/reference refresh (master pipeline step) |

> **Execution reality (trial capacity):** Spark & SQL kernels hit the 430 throttle, so every step above
> actually **runs on the pure-Python kernel** — `deltalake` over OneLake `abfss://`, `azure-kusto-data` for
> the eventhouse, `pyodbc`+ODBC 18 for the warehouse. The `_spark` notebooks are the production-idiom
> variants (V-Order, broadcast joins, MERGE, partitionBy) shipped for the demo but **not run**.


## 1. Why the lakehouse exists (and how it differs from Flow A)

| Concern | Flow A — Eventhouse (hot) | Flow B — Lakehouse (cold/historic) |
|---|---|---|
| Latency | seconds (streaming) | minutes→hours (scheduled batch) |
| Retention | 7 d / 90 d KQL | open Delta history (cheap OneLake) |
| Shape | wide event table + MVs | medallion star schema (fact + agg) |
| Serving | KQL dashboard `dsh_mch` | Direct Lake `sm_mch_historic` → Power BI |
| Question | "what is happening **now**?" | "how has the fleet behaved **over time**?" |

Both read from the **same** classified table `kdb_mch.tbl_mch_clean` — Flow A consumes it live; Flow B
**replicates a lean, curated slice** of it into the lake on a schedule. One source of truth, two serving stores.

## 2. The lean-bronze BAD/NULL filter (rationale)

The bronze ingest (`nb_mch_lh_ingest`) pulls from `tbl_mch_clean` but **drops `quality_flag IN ('BAD','NULL')`
at the source query**:

```kusto
tbl_mch_clean
| where ingest_ts > <bronze_watermark>
| where quality_flag !in ('BAD','NULL')      // <-- lean-bronze filter
| project event_id, event_ts, ingest_ts, lag_sec, plant_id, line_id, machine_id, status,
          temperature, vibration, pressure, rpm, power_kw, quality_flag,
          is_alert, is_anomaly, is_late, dq_reason, schema_version
| order by ingest_ts asc
```

**Why filter in the eventhouse, not the lake?**
- **Rejects already have a home.** BAD (out-of-range) and NULL (missing field) rows stay in `tbl_mch_clean`
  and are fully queryable via the Flow-A function **`fn_mch_errors()`** (the DQ/error feed that drives the
  Activator email-on-error path). They are *not lost* — they are simply not replicated.
- **Lean medallion.** Bronze→silver→gold→Power BI carries only data that can become a fact row. The lake
  doesn't pay storage/compute to copy garbage it would only re-reject downstream.
- **Clear contract.** "Bronze = ingestible telemetry; the eventhouse = system-of-record incl. rejects."
  Silver still keeps its **own** DLQ (`slv.dlq`) for the *lake-specific* rejects that only the lake can
  detect (unknown master refs, bad schema_version, stale-beyond-grace) — see §5.


## 3. Schema layout (`lh_mch`, schema-enabled Lakehouse)

OneLake path convention: `abfss://{ws}@onelake.dfs.fabric.microsoft.com/{lh}.Lakehouse/Tables/<schema>/<table>`.

| Schema | Table | Grain | Purpose |
|---|---|---|---|
| `brz` | `evt` | per event (append) | raw lean landing from eventhouse (+ `bronze_load_ts`) |
| `slv` | `evt` | per event (MERGE on `event_id`) | curated, de-duped, validated good rows (+ `silver_load_ts`) |
| `slv` | `dlq` | per rejected event | lake dead-letter queue (`dq_category`,`dq_reason`,`silver_load_ts`) |
| `gld` | `ft_telemetry` | per event | enriched fact: telemetry + machine/line/plant attrs + `date_sk` |
| `gld` | `agg_machine_daily` | machine × day | availability%, alert/anomaly/late rate, MTBF-ish |
| `ctl` | `wm` | per (layer, source) | incremental high-watermarks |
| `ctl` | `freshness` | per check (append) | freshness SLA measurement log |

### `brz.evt` columns
`event_id, event_ts, ingest_ts, lag_sec, plant_id, line_id, machine_id, status, temperature, vibration,
pressure, rpm, power_kw, quality_flag, is_alert, is_anomaly, is_late, dq_reason, schema_version,
bronze_load_ts`

### `slv.evt` columns (DLQ columns dropped, `silver_load_ts` added)
`event_id, event_ts, ingest_ts, lag_sec, plant_id, line_id, machine_id, status, temperature, vibration,
pressure, rpm, power_kw, quality_flag, is_alert, is_anomaly, is_late, schema_version, silver_load_ts`

### `gld.ft_telemetry` adds (from the dim join)
`event_date, date_sk, machine_name, machine_type, criticality, manufacturer, model, firmware_version,
rated_power_kw, max_rpm, max_temp_c, line_name, product_category, plant_name, region, country, city,
dim_matched, gold_load_ts`

### `gld.agg_machine_daily` columns
`machine_id, date_sk, machine_name, line_id, plant_id, plant_name, region, criticality, events, alerts,
anomalies, late_events, running, down, avg_temp, max_temp, avg_vib, max_vib, avg_rpm, avg_power,
first_ts, last_ts, availability_pct, alert_rate_pct, anomaly_rate_pct, late_pct, mtbf_hours, gold_load_ts`


## 4. Bronze ingest — `nb_mch_lh_ingest`  (SCHEDULED, incremental, idempotent)

- **Trigger:** scheduled (the `pl_mch_lh_master` pipeline, Daily 00:00 UTC; can run every 4–6 h).
- **Incremental key:** `ctl.wm` row `(layer='bronze', source='tbl_mch_clean')` holds the max `ingest_ts`
  already loaded. The pull is `where ingest_ts > watermark` (eventhouse `ingest_ts = ingestion_time()`),
  so each run only fetches newly-arrived clean rows. First run = epoch ⇒ full load.
- **Lean filter:** drops `quality_flag IN ('BAD','NULL')` (§2).
- **Write:** `write_deltalake(..., mode="append")` into `brz.evt`, plus an audit `bronze_load_ts`.
- **Idempotent:** if no new rows ⇒ no-op, watermark unchanged. Append is safe because the watermark only
  ever advances past rows actually written.
- **Signal:** writes `Files/_lh_ingest_signal.txt` ("ingestion started … | loaded N rows … | watermark=…")
  as a stand-in for the email/Activator notification (the real alert is a pipeline Office365-Outlook
  email activity or a Reflex rule on this marker — both portal-gated).

**Verified run:** loaded 5930 new rows; **brz.evt total = 11344**.

## 5. Bronze → Silver — `nb_mch_lh_bronze_silver`  (incremental MERGE + DLQ)

- **Incremental HWM:** `ctl.wm (layer='silver', source='brz.evt')` on **`event_ts`**.
- **24 h re-scan grace:** reprocesses from `hwm − 24h` so **late rows that already landed in bronze** get
  re-evaluated. Safe because of the MERGE (below).
- **De-dup within batch:** latest `ingest_ts` per `event_id` wins.
- **Idempotent MERGE on `event_id`** into `slv.evt`
  (`when_matched_update_all().when_not_matched_insert_all()`) — re-runs/backfills never duplicate.
- **DLQ routing** → `slv.dlq` with `dq_category` + `dq_reason`:

| `dq_category` | Rule |
|---|---|
| `UNKNOWN_REF` | plant/line/machine id missing → cannot resolve a master reference |
| `NULL` | any required vital (`temperature/vibration/pressure/rpm/power_kw`) null |
| `BAD` | `quality_flag='BAD'` or `dq_reason` contains `OUT_OF_RANGE` |
| `SCHEMA` | `schema_version` ∉ `{v1, v1.0, v1.1, 1.0, 1}` |
| `STALE` | **late-event grace**: `event_ts < hwm − 24h` (truly stale backlog) |

Good rows (no `dq_category`) MERGE into `slv.evt`; rejects MERGE into `slv.dlq`. The new silver
watermark = `max(hwm, max(event_ts) in batch)`.

> **Note on the two filters:** BAD/NULL are *already* dropped at bronze (§2), so in steady state silver's
> BAD/NULL branches mostly catch reprocessed edge cases; silver's real, unique value is `UNKNOWN_REF` /
> `SCHEMA` / `STALE` — rejects only the lake can determine.

**Verified run (first load):** good→`slv.evt`=5019, dlq→`slv.dlq`=383 (NULL 228, BAD 155).
**Cumulative after orchestrated runs: `slv.evt`=10006, `slv.dlq`=726.**


## 6. Silver → Gold — `nb_mch_lh_gold`  (CROSS-STORE dim reuse)

The lakehouse does **not** re-model the dimensions — it **reuses** the warehouse master data (`wh_mch`,
built by Flow-A/dim side). This is a genuine **cross-store join** (Lakehouse Delta ⋈ Warehouse T-SQL):

1. `pyodbc` (ODBC Driver 18 + `getToken("pbi")`) reads the **current** SCD rows
   (`WHERE scd_is_current=1`) of `dim.DIM_MACHINE`, `dim.DIM_LINE`, `dim.DIM_PLANT` into pandas.
2. Build the hierarchy `machine → line → plant` (dim is source of truth for line/plant).
3. **Left-join** `slv.evt` to the dim on `machine_id` (left ⇒ keep events even if a dim ref is missing;
   `dim_matched` flags whether a current machine row was found).
4. Add `event_date`, `date_sk` (`YYYYMMDD` int) and write **`gld.ft_telemetry`** (`mode="overwrite",
   schema_mode="overwrite"` — deterministic full rebuild from silver).
5. Roll up **`gld.agg_machine_daily`** per `machine_id × date_sk`:
   - `availability_pct` = running / events (running = status ∉ {IDLE, MAINTENANCE, OFFLINE})
   - `alert_rate_pct`, `anomaly_rate_pct`, `late_pct`
   - `mtbf_hours` (MTBF-ish) = observed span (h) / (alerts+anomalies); no failures ⇒ whole span.

**Verified run:** `gld.ft_telemetry`=10006 (= silver good rows), `gld.agg_machine_daily`=**32** machine-days.

**Why current-rows-only?** The fact is "what the machine *is* now" enrichment for trend BI. Point-in-time
SCD2 history lives in the warehouse (`dim.*` brackets) for audit; gold deliberately denormalizes the
**current** descriptor onto each event so Direct Lake stays single-pass and fast.

## 7. Watermarks & freshness — `ctl.wm` + `ctl.freshness`

### `ctl.wm` (incremental control)
One row per `(layer, source)`: `layer, source, hwm_value, rows_last_load, updated_at`. Read/written by the
helper pair `read_wm` / `write_wm` (overwrite-the-matching-row pattern) in every layer notebook.

| layer | source | meaning |
|---|---|---|
| `bronze` | `tbl_mch_clean` | max `ingest_ts` pulled from the eventhouse |
| `silver` | `brz.evt` | max `event_ts` promoted to silver |

### `ctl.freshness` (SLA log) — `nb_mch_lh_freshness`
Computes `now − max(gld.ft_telemetry.event_ts)` in minutes vs a **1440-min (24 h) POC SLA**, **appends**
a row `(checked_at, gold_max_event_ts, delay_minutes, threshold_minutes, sla_breach)`, and on breach writes
the **pipeline-delay** marker `Files/_lh_freshness_signal.txt` (stands in for the Activator/email alert —
real alert = Activator rule on `ctl.freshness.sla_breach`, or a pipeline email-on-condition).
**Verified:** delay 27.72 min ⇒ `breach=False` (within SLA).


## 8. Maintenance, backfill & orchestration

### `nb_mch_lh_maint` — OPTIMIZE + VACUUM
Across `brz.evt, slv.evt, slv.dlq, gld.ft_telemetry, gld.agg_machine_daily, ctl.wm`:
`DeltaTable.optimize.compact()` (= OPTIMIZE bin-packing of the small files that incremental append/MERGE
churn produces) then `vacuum(retention_hours=168)` (7-day tombstone purge). Keeps the lake healthy.

### `nb_mch_lh_orchestrator` — the Flow-B chain (+ backfill)
Runs the children in order via `notebookutils.notebook.run` (same Python capacity):
`ingest → [rewind] → bronze_silver → gold → maint → freshness`.
- **Parameterized:** `backfill_hours` (overridable `-P backfill_hours:int=N`). When `>0`, it **rewinds the
  silver watermark** by N hours before the silver step, so a window of already-bronzed events is
  **re-MERGEd** — the backfill capability, made safe by the idempotent `event_id` MERGE.
- **Verified roll-up:** brz=11344 · slv.evt=10006 · slv.dlq=726 · gld.ft_telemetry=10006 · agg=32.

## 9. The three pipelines + schedule + alerts (`pl_mch_lh_*`)

A real pipeline hierarchy (not just notebook-runners):

```
pl_mch_lh_master (parent, SCHEDULED Daily 00:00 UTC)
  ├─ Wait (ingest window)  → Set mode='full'
  ├─ RefreshDataflow  df_mch_dimrefresh   (Dataflow Gen2 — low-code dim/reference refresh)
  ├─ InvokePipeline   pl_mch_lh_ingest    (waitOnCompletion)
  ├─ IfCondition  mode=='full' → InvokePipeline pl_mch_lh_medallion
  ├─ Web  "Master run complete"            (on Succeeded)
  └─ Web  "MASTER PIPELINE FAILED"         (on Failed)

pl_mch_lh_ingest (child)
  ├─ Web "ingestion started" → Notebook nb_mch_lh_ingest (retry 2 / 120 s, timeout 1 h)
  ├─ Web "ingestion loaded OK"             (on Succeeded)
  └─ Web "INGEST FAILED"                   (on Failed)

pl_mch_lh_medallion (child)
  ├─ Notebook bronze_to_silver (retry 2) → Wait 30 s → Notebook silver_to_gold (retry 2)
  ├─ Notebook optimize_vacuum → Notebook freshness_check
  └─ Web "MEDALLION FAILED"                (on any Failed)
```

- **Schedule:** Daily **00:00 UTC** on the master, created via `…/jobs/Pipeline/schedules`.
- **Alerts:** each stage has success/failure **Web** activities → a Teams/Logic-App/Power-Automate webhook
  (`NOTIFY_URL`, placeholder). Swap any Web for an **Office365 Outlook 'Send email'** activity (needs a
  portal connection) for native email. Notebook signal-markers (§4/§7) are the alternate alert source for
  an Activator/Reflex rule.

## 10. Serving layer — Direct Lake `sm_mch_historic` + report `rpt_mch_historic`

### Semantic model — `nb_mch_hist_model`
`semantic-link-labs` `directlake.generate_direct_lake_semantic_model(source_type='Lakehouse',
use_sql_endpoint=True)` over the **gold** tables `gld.ft_telemetry` + `gld.agg_machine_daily`, then TOM
adds measures and sets **`DirectLakeBehavior = DirectLakeOnly`** (no DirectQuery fallback).

**Measures (verified created):** `Event Count, Alert Count, Anomaly Count, Late Count, Alert Rate %,
Anomaly Rate %, Late %, Availability %, Avg Temp, Avg Vibration, Avg MTBF (hrs)`.
`Availability % = DIVIDE(CALCULATE(COUNTROWS(ft_telemetry), NOT(status IN {"IDLE","MAINTENANCE","OFFLINE"})),[Event Count])`.

**Refresh (Direct Lake reframe):** weekdays **06:00 / 12:00 / 18:00 UTC**.

### Report — `nb_mch_hist_model + report builder` → `rpt_mch_historic`
A thin report **bound** to `sm_mch_historic` is created headless via
`report.create_report_from_reportjson` (verified built); the **visual layout** (visualContainers JSON) is
portal-authored in this env, so the notebook also emits the exact page/visual spec to point-and-click:

| Page | Visuals |
|---|---|
| **Fleet Overview** | Cards [Event Count]/[Availability %]/[Alert Rate %]/[Anomaly Rate %] · column region×[Event Count] · bar plant_name×[Availability %] |
| **Machine Health** | Table machine_id/name/criticality + rate measures + [Avg MTBF (hrs)] · scatter [Avg Vibration]×[Avg Temp] by machine |
| **Daily Trend** | Lines `date_sk`×[Availability %] and ×[Alert/Anomaly Rate %] · column `date_sk`×[Late %] |

## 11. Silver-query utility (ad-hoc historic querying)

Because `lh_mch` is a **schema-enabled** Lakehouse, every `slv`/`gld` Delta table is exposed through the
Lakehouse **SQL analytics endpoint** for ad-hoc T-SQL — the curated query surface over silver/gold
*without* touching the eventhouse:

```sql
-- "good curated telemetry" (silver)
SELECT TOP 100 * FROM slv.evt ORDER BY event_ts DESC;
-- DLQ triage: why was data rejected by the lake?
SELECT dq_category, COUNT(*) n FROM slv.dlq GROUP BY dq_category ORDER BY n DESC;
-- daily fleet KPIs (gold)
SELECT date_sk, AVG(availability_pct) avail, SUM(alerts) alerts
FROM gld.agg_machine_daily GROUP BY date_sk ORDER BY date_sk;
```

The same tables are read in-notebook via `deltalake` over `abfss://` (the executable Python path) — the
`read_wm`/`exists`/`tpath` helpers in every `nb_mch_lh_*` notebook are the reusable query primitives.
DLQ counts (`slv.dlq` by `dq_category`) are the lake's data-quality KPI, paralleling Flow-A's
`fn_mch_errors()`.

## 12. OneLake-availability alternative (zero-copy mirror) — and its trade-off

Today Flow B **physically replicates** a lean slice of `tbl_mch_clean` into the lake (the scheduled ingest).
An alternative is to **turn on OneLake availability** for the KQL database: the eventhouse then continuously
**mirrors its tables to OneLake as Delta**, so the lake can read them **zero-copy** — *no* `nb_mch_lh_ingest`
job, *no* `brz.evt`, *no* bronze watermark; silver would read the mirrored Delta directly.

| | Scheduled replicate (current) | OneLake-availability mirror (alternative) |
|---|---|---|
| Copy | physical (we own `brz.evt`) | zero-copy (eventhouse-managed Delta) |
| Filter | lean BAD/NULL filter at ingest | **no** ingest-time filter — mirrors *all* rows (incl. rejects); filter must move to silver |
| Latency | scheduled (batch) | near-continuous (mirror cadence) |
| Control | full (watermark, audit col, shaping) | less (schema/cadence set by the eventhouse) |
| Cost | extra storage + compute for the copy | minimal — reuse hot store's Delta |
| Coupling | decoupled; lake survives eventhouse changes | coupled to the eventhouse table layout & lifecycle |

**Trade-off:** the mirror is cheaper and fresher but gives up the **lean-bronze shaping** (you mirror BAD/NULL
too, pushing all DQ rejection into silver) and couples the lake to the eventhouse's table contract. The POC
keeps the **explicit scheduled replicate** for control + the lean-bronze contract; OneLake availability is the
documented optimization once the schema is frozen.


## 13. Dual-code: PySpark / Spark-SQL presentation variants

Per the project rule, each medallion step ships **two** notebooks: the runnable **Python** one (above) and a
**`_spark`** production-idiom variant. The `_spark` notebooks are **imported but not run** (trial Spark =
430 throttle); they exist to show the "proper" big-data form on screen.

| Runnable (Python kernel) | Presentation (`synapse_pyspark`, gated) | Idiom demonstrated |
|---|---|---|
| `nb_mch_lh_ingest` | `nb_mch_lh_ingest_spark` | Kusto Spark connector read + V-Order + OPTIMIZE |
| `nb_mch_lh_bronze_silver` | `nb_mch_lh_bronze_silver_spark` | window de-dup + `DeltaTable.merge` + DLQ + optimizeWrite |
| `nb_mch_lh_gold` | `nb_mch_lh_gold_spark` | `synapsesql` cross-store read + **broadcast** dim join + `partitionBy(date_sk)` |
| `nb_mch_lh_maint` | `nb_mch_lh_maint_spark` | Spark-SQL `OPTIMIZE … ZORDER BY (machine_id)` + `VACUUM … RETAIN 168 HOURS` |

## 14. Full lineage

```
kdb_mch.tbl_mch_clean                                  (eventhouse, classified)
   │  good rows  ─ nb_mch_lh_ingest ─ where ingest_ts>wm  AND quality_flag∉{BAD,NULL}
   │              (ctl.wm bronze/tbl_mch_clean)
   ▼
brz.evt  (lh_mch, append, +bronze_load_ts)
   │  nb_mch_lh_bronze_silver ─ event_ts HWM (−24h rescan) ─ MERGE on event_id
   │              (ctl.wm silver/brz.evt)
   ├─►  slv.dlq   (UNKNOWN_REF / NULL / BAD / SCHEMA / STALE)
   ▼
slv.evt  (curated good, +silver_load_ts)
   │  nb_mch_lh_gold ─ LEFT JOIN on machine_id  ◄── wh_mch.dim.DIM_MACHINE/LINE/PLANT (scd_is_current=1, pyodbc)
   ▼
gld.ft_telemetry ──rollup(machine_id×date_sk)──► gld.agg_machine_daily
   │  nb_mch_hist_model ─ Direct Lake (DirectLakeOnly), SQL endpoint
   ▼
sm_mch_historic  (semantic model + measures, reframe weekdays 06/12/18 UTC)
   │  nb_mch_hist_model + report builder
   ▼
rpt_mch_historic  (Power BI: Fleet Overview · Machine Health · Daily Trend)

ctl.wm        ← read/written by ingest + bronze_silver (incremental control)
ctl.freshness ← nb_mch_lh_freshness (SLA log; breach ⇒ pipeline-delay marker)

Orchestration: pl_mch_lh_master (Daily 00:00 UTC) → df_mch_dimrefresh → pl_mch_lh_ingest → pl_mch_lh_medallion
               (alerts via Web→webhook / Office365 email; notebook signal markers feed Activator)
```

| Source | Transform / item | Target | Key / mode |
|---|---|---|---|
| `tbl_mch_clean` | `nb_mch_lh_ingest` | `brz.evt` | `ingest_ts` watermark, append, BAD/NULL filtered |
| `brz.evt` | `nb_mch_lh_bronze_silver` | `slv.evt`, `slv.dlq` | `event_ts` HWM, MERGE on `event_id`, 24h grace→STALE |
| `slv.evt` + `wh_mch.dim.*` | `nb_mch_lh_gold` | `gld.ft_telemetry` | LEFT JOIN `machine_id`, overwrite |
| `gld.ft_telemetry` | `nb_mch_lh_gold` | `gld.agg_machine_daily` | groupby `machine_id × date_sk` |
| `gld.*` | `nb_mch_hist_model` | `sm_mch_historic` | Direct Lake (DirectLakeOnly) |
| `sm_mch_historic` | `nb_mch_hist_model + report builder` | `rpt_mch_historic` | live-bound report |
| `gld.ft_telemetry` | `nb_mch_lh_freshness` | `ctl.freshness` | append SLA log |

## 15. Operate / rebuild
```bash
# full chain (build then run via fab job; or schedule pl_mch_lh_master)
.venv/bin/fab job run "ws_mch_iot.Workspace/nb_mch_lh_orchestrator.Notebook" --timeout 360
# backfill a window (re-MERGE already-bronzed events, idempotent on event_id)
#   -P backfill_hours:int=48
# rebuild the serving layer only
.venv/bin/fab job run "ws_mch_iot.Workspace/nb_mch_hist_model.Notebook" --timeout 360
```
See the Real-Time doc (`nb_mch_rt_docs`) for the eventhouse/hot side and the Warehouse doc
(`nb_mch_wh_docs`) for the SCD2 master-data dimensions this gold layer joins to.


## 16. Star-schema serving (current) + verified end-to-end

**Serving model `sm_mch_historic` is now a proper STAR** (built by nb_mch_hist_model / nb_mch_hist_star / nb_mch_hist_security):
- 6 Direct Lake tables on `gld`: facts `ft_telemetry` + `agg_machine_daily`; dims `dim_machine`(31) `dim_line`(18) `dim_plant`(10) `dim_date`(calendar).
- **8 relationships** (both facts -> all 4 dims, many:1, single-direction) -> one slicer filters fact **and** daily-agg together.
- `dim_date` = model Date table with a **Year > Quarter > Month > Day** hierarchy (time-intelligence / drill-down).
- 21 measures (incl. OEE% proxy, MTBF, rates), **RLS** (PlantManager_APAC, FleetViewer) + **OLS** (Operator hides manufacturer/model/firmware/lag_sec), DirectLakeOnly, weekday 06/12/18 UTC reframe.
- Report `rpt_mch_historic` = 3 pages / 21 visuals on the star (calendar drill axis + calendar slicer).

**Verified END-TO-END (2026-06-20):** streamed new telemetry -> eventhouse `tbl_mch_clean` -> `nb_mch_lh_orchestrator` (ingest->bronze->silver->gold->maint->freshness). Result: brz.evt 11107->11403, slv.evt 11096->11383, **slv.dlq stayed 0** (bronze BAD/NULL filter), gld.ft_telemetry 11096->11383. Direct Lake **reframe** then returned `[Event Count]=11383` (exactly matching gold), Alert Rate 59%, Availability 85.8%, 31 machines -> **the report reflects new eventhouse data through the whole chain.**

Schedule: `pl_mch_lh_master` Daily 00:00 UTC runs this chain unattended; freshness writes `Files/_lh_freshness_signal.txt` on SLA breach; Activator `act_mch_lh` / pipeline Web activities raise the alerts.
